In [40]:
import cv2
import numpy as np
import os
from skimage.feature import graycomatrix, graycoprops
from skimage.feature import local_binary_pattern
from sklearn.cluster import KMeans
import pandas as pd

# ===== SETTINGS =====
DATASET_PATH = "roi_output"

GLCM_CSV = "glcm_features.csv"
KMEANS_CSV = "kmeans_features.csv"

glcm_list = []
kmeans_list = []

labels_glcm = []
labels_kmeans = []

# ===== LOOP =====
for class_name in os.listdir(DATASET_PATH):

    class_path = os.path.join(DATASET_PATH, class_name)
    if not os.path.isdir(class_path):
        continue

    for img_name in os.listdir(class_path):

        img_path = os.path.join(class_path, img_name)

        img = cv2.imread(img_path, 0)
        if img is None:
            continue

        img = cv2.resize(img, (128, 128))

        # =========================
        # 🔹 GLCM + LBP FEATURES
        # =========================

        # ----- GLCM -----
        glcm = graycomatrix(img, distances=[1], angles=[0, np.pi/4],
                            levels=256, symmetric=True, normed=True)

        contrast = graycoprops(glcm, 'contrast').flatten()
        energy = graycoprops(glcm, 'energy').flatten()
        homogeneity = graycoprops(glcm, 'homogeneity').flatten()
        correlation = graycoprops(glcm, 'correlation').flatten()

        glcm_features = np.hstack([contrast, energy, homogeneity, correlation])


        # ----- LBP -----
        radius = 1
        n_points = 8 * radius

        lbp = local_binary_pattern(img, n_points, radius, method='uniform')

        # LBP histogram
        lbp_hist, _ = np.histogram(
            lbp.ravel(),
            bins=np.arange(0, n_points + 3),
            range=(0, n_points + 2)
        )

        # normalize
        lbp_hist = lbp_hist.astype("float")
        lbp_hist /= (lbp_hist.sum() + 1e-6)

        # =========================
        # 🔹 HISTOGRAM (ADD TO GLCM)
        # =========================
        mean = np.mean(img)
        std = np.std(img)

        hist, _ = np.histogram(img.flatten(), bins=256, range=[0,256])
        prob = hist / np.sum(hist)
        entropy = -np.sum(prob * np.log2(prob + 1e-10))

        hist_features = [mean, std, entropy]

        # =========================
        # 🔹 FINAL COMBINED FEATURES
        # =========================
        glcm_lbp_features = np.hstack([
            glcm_features,
            lbp_hist,
            hist_features   #  added
        ])

        glcm_list.append(glcm_lbp_features)
        labels_glcm.append(class_name)


        # =========================
        #  K-MEANS FEATURES
        # =========================
        pixels = img.reshape(-1, 1)

        kmeans = KMeans(n_clusters=2, random_state=0).fit(pixels)

        centers = np.sort(kmeans.cluster_centers_.flatten())
        c1, c2 = centers

        distance = abs(c2 - c1)

        labels_k = kmeans.labels_
        counts = np.bincount(labels_k)

        p1 = counts[0] / len(labels_k)

        kmeans_features = [c1, c2, distance, p1]

        kmeans_list.append(kmeans_features)
        labels_kmeans.append(class_name)



# ===== SAVE GLCM CSV =====
df_glcm = pd.DataFrame(glcm_list)
df_glcm['label'] = labels_glcm
df_glcm.to_csv(GLCM_CSV, index=False)


# ===== SAVE KMEANS CSV =====
df_kmeans = pd.DataFrame(kmeans_list)
df_kmeans['label'] = labels_kmeans
df_kmeans.to_csv(KMEANS_CSV, index=False)


print("2 CSV files saved ")

2 CSV files saved 


In [72]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Models
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


def run_experiment(file_path, dataset_name):
    print(f"\n===== {dataset_name} DATASET =====")

    # Load data
    df = pd.read_csv(file_path)

    X = df.drop(columns=['label'])
    y = df['label']

    # Normalize
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Models
    models = {
        "SVM": SVC(kernel='rbf', C=1, gamma="scale"),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RandomForest": RandomForestClassifier(n_estimators=50, n_jobs=-1),
        "LogisticRegression": LogisticRegression(max_iter=500),
        "DecisionTree": DecisionTreeClassifier(max_depth=5),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=100)
    }

    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        print(f"{name}: Acc={acc:.4f}, Prec={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

        results.append((name, acc, precision, recall, f1))


# ===== RUN FOR BOTH DATASETS =====
run_experiment("glcm_features.csv", "GLCM")
run_experiment("kmeans_features.csv", "KMeans")


===== GLCM DATASET =====
SVM: Acc=0.8125, Prec=0.8165, Recall=0.8125, F1=0.8113
KNN: Acc=0.7292, Prec=0.7300, Recall=0.7292, F1=0.7293
RandomForest: Acc=0.7292, Prec=0.7373, Recall=0.7292, F1=0.7250
LogisticRegression: Acc=0.7500, Prec=0.7500, Recall=0.7500, F1=0.7500
DecisionTree: Acc=0.6667, Prec=0.6667, Recall=0.6667, F1=0.6667
ExtraTrees: Acc=0.7708, Prec=0.7740, Recall=0.7708, F1=0.7693

===== KMeans DATASET =====
SVM: Acc=0.5208, Prec=0.5289, Recall=0.5208, F1=0.5135
KNN: Acc=0.5208, Prec=0.5217, Recall=0.5208, F1=0.5210
RandomForest: Acc=0.5833, Prec=0.5825, Recall=0.5833, F1=0.5819
LogisticRegression: Acc=0.5000, Prec=0.4982, Recall=0.5000, F1=0.4983
DecisionTree: Acc=0.5000, Prec=0.4721, Recall=0.5000, F1=0.4273
ExtraTrees: Acc=0.5417, Prec=0.5403, Recall=0.5417, F1=0.5401


In [69]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# ===== SETTINGS =====
DATASET_PATH = "roi_output"
BATCH_SIZE = 16
IMG_SIZE = 128
EPOCHS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== TRANSFORM =====
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_data, test_data = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

# ===== MODELS =====
model_names = {
    "ResNet18": models.resnet18(pretrained=True),
    "ResNet50": models.resnet50(pretrained=True),
    "VGG16": models.vgg16(pretrained=True),
    "MobileNet": models.mobilenet_v2(pretrained=True),
    "EfficientNet": models.efficientnet_b0(pretrained=True)
}

results = []

for name, model in model_names.items():

    print(f"\nRunning {name}")

    # Modify last layer
    if "resnet" in name.lower():
        model.fc = nn.Linear(model.fc.in_features, 3)
    elif "vgg" in name.lower():
        model.classifier[6] = nn.Linear(4096, 3)
    elif "mobilenet" in name.lower():
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 3)
    elif "efficientnet" in name.lower():
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 3)

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # ===== TRAIN =====
    model.train()
    for epoch in range(EPOCHS):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # ===== TEST =====
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, pred = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (pred == labels).sum().item()

    acc = correct / total
    print(f"{name} Accuracy: {acc:.4f}")

    results.append((name, acc))


print("\n===== DL COMPARISON =====")
for name, acc in results:
    print(f"{name}: {acc:.4f}")

C:\Users\sunda\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\sunda\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\sunda\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\models\_utils.py:223: UserWa


Running ResNet18
ResNet18 Accuracy: 0.6042

Running ResNet50
ResNet50 Accuracy: 0.6875

Running VGG16
VGG16 Accuracy: 0.4792

Running MobileNet
MobileNet Accuracy: 0.7708

Running EfficientNet
EfficientNet Accuracy: 0.6667

===== DL COMPARISON =====
ResNet18: 0.6042
ResNet50: 0.6875
VGG16: 0.4792
MobileNet: 0.7708
EfficientNet: 0.6667


In [73]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ===== SETTINGS =====
DATASET_PATH = "roi_output"
BATCH_SIZE = 16
EPOCHS = 10
IMG_SIZE = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== TRANSFORMS =====
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ===== DATASET =====
dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ===== MODEL =====
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 3)   # 3 classes
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = CNN().to(device)

# ===== LOSS & OPTIMIZER =====
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ===== TRAINING =====
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss:.4f}")

# ===== EVALUATION =====
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total

print("\n===== CNN RESULT =====")
print(f"Accuracy: {accuracy:.4f}")

Epoch [1/10], Loss: 9.6259
Epoch [2/10], Loss: 7.7897
Epoch [3/10], Loss: 7.7622
Epoch [4/10], Loss: 7.0891
Epoch [5/10], Loss: 6.6107
Epoch [6/10], Loss: 6.4229
Epoch [7/10], Loss: 7.0245
Epoch [8/10], Loss: 7.1241
Epoch [9/10], Loss: 6.1916
Epoch [10/10], Loss: 5.7591

===== CNN RESULT =====
Accuracy: 0.6250
